# How to connect to Milvus from a notebook

启动 Milvus 服务器有多种不同的方法。

1. Milvus Lite 是一个本地 Python 服务器，可在 Jupyter 笔记本或 Google Colab 中运行，需要 pymilvus>=2.4.3。⛔️ 仅用于演示和本地测试
2. Zilliz 云免费版
3. Milvus 独立式 Docker 需要本地安装并运行 Docker
4. LangChain - 所有第三方适配器均使用 Milvus Lite
5. LlamaIndex - 所有第三方适配器均使用 Milvus Lite
6. Milvus Kubernetes 集群需要已部署并运行的 K8s 集群

💡对于生产工作负载，建议使用 Milvus 本地 Docker、Kubernetes 集群，或 Zilliz Cloud 上的完全托管 Milvus。

我将演示如何使用 Python SDK 进行连接。更多详细信息，请参见这个 Python 示例。

In [1]:
import sys
import os

print(sys.executable)
CUSTOM_CACHE = r'F:\Teewon\Milvue\models'
os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(CUSTOM_CACHE, 'transformers')

F:\Teewon\Milvue\.venv\Scripts\python.exe


In [2]:
from huggingface_hub import constants
print(constants.HF_HUB_CACHE)

F:\Teewon\Milvue\models\hub


## 1. Milvus Lite

Milvus Lite 是一个轻量级的 Python 服务器，可本地运行。它非常适合在笔记本电脑、Jupyter 笔记本或 Colab 上快速入门 Milvus。

⛔️请注意，Milvus Lite 仅用于演示，不适用于生产环境的工作负载

In [ ]:
#!python -m pip install -U pymilvus
import pymilvus
print(f"pymilvus:{pymilvus.__version__}")

In [ ]:
# Connect a client to the Milvus Lite server.
from pymilvus import MilvusClient
mc = MilvusClient("milvus_demo.db")

In [ ]:
# Create a collection.
COLLECTION_NAME = "MilvusDocs"
EMBEDDING_DIM = 256

# Milvus Lite uses the MilvusClient object.
if mc.has_collection(COLLECTION_NAME):
    mc.drop_collection(COLLECTION_NAME)
    print(f"Successfully dropped collection: `{COLLECTION_NAME}`")

# Create a collection with flexible schema and AUTOINDEX.
mc.create_collection(
    COLLECTION_NAME,
    EMBEDDING_DIM,
    consistency_level="Eventually",
    auto_id=True,
    overwrite=True,
)
print(f"Successfully created collection: `{COLLECTION_NAME}`")

In [ ]:
# Drop the collection
mc.drop_collection(COLLECTION_NAME)
print(f"Successfully dropped collection: `{COLLECTION_NAME}`")

## 2. Zilliz free tier

本部分使用 Zilliz 的免费版。如果您尚未注册，请先申请免费试用。

如果您已有 Zilliz 账户并希望使用免费版，只需在创建集群时选择“入门”选项即可。❤️‍🔥 **也就是说，所有人都可以享受免费版服务！**

- 每个账户可使用一个免费层级的集群。
- 每个免费层级集群最多可同时支持两个集合（将集合类比为数据库表，每个集合包含索引、模式和一致性级别）。
- 每个免费层级集合最多可支持100万个向量（可理解为数据库表中的行数）。

如果您的数据规模更大，我们建议选择按需付费的无服务器或企业版方案。免费层级和按需付费服务均由Zilliz管理，基于AWS、Google或Azure提供。企业版支持自建机房（BYOC）部署。

### 👩 Set up instructions for Zilliz

1. 从 cloud.zilliz.com，点击**“+ 创建Cluster”**
2. 选择集群的**“启动”**选项，然后点击**“下一步：创建Collection”**
3. 为您的Collection命名一个**Collection Name**，然后点击**“创建Collection and Cluster”**
4. 从集群页面，

    - 复制集群URI并保存到本地
    - 复制你的集群API密钥，务必保密！

5. 将 API KEY 添加到您的环境变量中
6. 在 Jupyter 中，你还需要一个 .env 文件（与笔记本文件位于同一目录），其中包含类似以下内容的行：

    ZILLIZ_API_KEY=value
7. 在您的代码中，连接到您的 Zilliz 集群，参见下方代码示例

In [ ]:
import os
from pymilvus import (connections, MilvusClient, utility)
TOKEN = os.getenv("ZILLIZ_API_KEY")

# Connect to Zilliz cloud using endpoint URI and API key TOKEN.
CLUSTER_ENDPOINT="https://in03-xxxx.api.gcp-us-west1.zillizcloud.com:443"
CLUSTER_ENDPOINT="https://in03-8bc9fd463236b1a.api.gcp-us-west1.zillizcloud.com:443"

connections.connect(
  alias='default',
  uri=CLUSTER_ENDPOINT,
  token=TOKEN,
)

# Check if the server is ready and get collection name.
print(f"Type of server: {utility.get_server_version()}")

In [ ]:
COLLECTION_NAME = "movies"
EMBEDDING_DIM = 256

# Use no-schema Milvus client uses flexible json key:value format.
# https://milvus.io/docs/using_milvusclient.md
mc = MilvusClient(
    uri=CLUSTER_ENDPOINT,
    token=TOKEN)

# Check if collection already exists, if so drop it.
has = utility.has_collection(COLLECTION_NAME)
if has:
    drop_result = utility.drop_collection(COLLECTION_NAME)
    print(f"Successfully dropped collection: `{COLLECTION_NAME}`")

# Create a collection with flexible schema and AUTOINDEX.
mc.create_collection(COLLECTION_NAME,
                     EMBEDDING_DIM,
                     consistency_level="Eventually",
                     auto_id=True,
                     overwrite=True,
                    )
print(f"Successfully created collection: `{COLLECTION_NAME}`")

In [ ]:
# Drop collection
utility.drop_collection(COLLECTION_NAME)

# Disconnect from the server.
try:
  connections.disconnect(alias="default")
  print("Successfully disconnected from the server.")
except:
  pass

## 3. Milvus standalone Docker

本部分使用 Docker 部署的 Milvus 独立实例
> ⛔️ 请确保正确安装 pymilvus 的版本以及 server yml 文件。所有版本（主版本和次版本）必须完全匹配。

1. 安装 Docker
2. 启动您的 Docker Desktop
3. 下载最新的 docker-compose.yml（或运行 wget 命令，将版本号替换为当前使用的版本）
> wget https://github.com/milvus-io/milvus/releases/download/v2.4.0-rc.1/milvus-standalone-docker-compose.yml -O docker-compose.yml

4. 从终端：
    - 进入保存 .yml 文件的目录（通常与本笔记本在同一目录）
    - docker compose up -d
    - 通过终端或 Docker Desktop 验证容器是否正在运行

5. 从代码中（参见下方笔记本代码）：
    - 导入 milvus
    - 连接到本地 milvus 服务器

In [ ]:
import pymilvus, time
from pymilvus import (connections, MilvusClient, utility)
print(f"Pymilvus: {pymilvus.__version__}")

In [ ]:
# Start Milvus standalone on docker, running quietly in the background.
# !docker compose up -d

# Verify which local port the Milvus server is listening on
# !docker ps -a #19530/tcp

In [ ]:
# Connect to the local server
connection=connections.connect(
    alias="default",
    host="localhost",
    port=19530,
)

# Get server version
print(utility.get_server_version())

In [ ]:
# Use no-schema Milvus client uses flexible json key:value format.
mc=MilvusClient(connections=connection)

COLLECTION_NAME="movies"
EMBEDDING_DIM=256

# Check if collection already exists, if so drop it
has=utility.has_collection(COLLECTION_NAME)
if has:
    drop_result=utility.drop_collection(COLLECTION_NAME)
    print(f"Successfully dropped collection: `{COLLECTION_NAME}`")

# Create a collection with flexible schema and AUTOINDEX
mc.create_collection(
    COLLECTION_NAME,
    EMBEDDING_DIM,
    consistency_level="Eventually",
    auto_id=True,
    overwrite=True,
)
print(f"Successfully created collection: `{COLLECTION_NAME}`")

In [ ]:
# Stop local milvus
!docker compose down

# Disconnect from the server
try:
    connections.disconnect(alias="default")
    print(f"Successfully disconnected from the server")
except:
    pass

## LangChain

所有第三方适配器均使用Milvus Lite

LangChain API 会隐藏将原始非结构化数据转换为向量并存储到 Milvus 中的许多步骤。
- LangChain 文档
- Milvus 文档

LangChain 默认值：
- collection_name: LangChainCollection
- schema: ['pk', 'source', 'text', 'vector']
- auto_id: True
- {'index_type': 'HNSW', 'metric_type': 'L2', 'params': {'M': 8, 'efConstruction': 64}}
- consistency_level: 'Session'
- overwrite: False

In [ ]:
#!python -m pip install -U langchain_community unstructured langchain-milvus langchain-huggingface

In [3]:
# 取消注释以从本地目录读取网页文档。

# Read docs into LangChain
from langchain_community.document_loaders import DirectoryLoader

# Load HTML files from a local directory
path="RAG/rtdocs_new/"
loader=DirectoryLoader(path,glob='*.html')
docs=loader.load()

num_documents=len(docs)
print(f"loaded {num_documents} documents")

C:\Users\Administrator\AppData\Local\Temp\ipykernel_40536\4082896512.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader


loaded 22 documents


In [4]:
# Inspect the first document
import pprint
print(f"length doc: {len(docs[0].page_content)}")
pprint.pprint(docs[0].page_content.replace("\n", "")[:100])

length doc: 1964
('milvus-logoDocsBlogCommunityStars0Homev2.4.xAbout MilvusGet '
 'StartedConceptsArchitectureOverviewStora')


In [5]:
from langchain_milvus import Milvus
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import time, pprint

# Define the embedding model
model_name="BAAI/bge-large-en-v1.5"
model_kwargs={'device':'cpu'}
encode_kwargs={'normalize_embeddings':True}
embed_model=HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

sample_embedding=embed_model.embed_query("dimension test")
EMBEDDING_DIM=len(sample_embedding)
print(f"EMBEDDING_DIM: {EMBEDDING_DIM}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

EMBEDDING_DIM: 1024


In [6]:
# Chunks
text_splitter=RecursiveCharacterTextSplitter(chunk_size=512,chunk_overlap=51)

# Create a Milvus collection from the documents and embeddings
start_time=time.time()
docs=text_splitter.split_documents(docs)
vectorstore=Milvus.from_documents(
    documents=docs,
    embedding=embed_model,
    connection_args={
        "uri":"milvus_demo.db"},
    # Override LangChain default values for Milvus
    consistency_level="Eventually",
    drop_old=True,
    index_params={
        "metric_type": "COSINE",
        "index_type":"AUTOINDEX",
        "params": {},}
)
end_time=time.time()
print(f"Created Milvus collection from {len(docs)} docs in {end_time-start_time:.2f} seconds")

Created Milvus collection from 742 docs in 340.28 seconds


In [7]:
# Describe the collection.
print(f"collection_namme: {vectorstore.collection_name}")
print(f"schema: {vectorstore.fields}")
print(f"auto_id: {vectorstore.auto_id}")
pprint.pprint(vectorstore.index_params)
pprint.pprint(f"consisitency_level: {vectorstore.consistency_level}")
vectorstore.drop_old=True
pprint.pprint(f"drop_old: {vectorstore.drop_old}")

collection_namme: LangChainCollection
schema: ['text', 'pk', 'vector', 'source']
auto_id: True
{'index_type': 'AUTOINDEX', 'metric_type': 'COSINE', 'params': {}}
'consisitency_level: Eventually'
'drop_old: True'


In [8]:
# Delete the Milvus collection
del vectorstore

## LlamaIndex

所有第三方适配器均使用Milvus Lite。

LlamaIndex APIs 会隐藏将原始非结构化数据转换为向量并存储到 Milvus 中的许多步骤。

- LlamaIndex 文档
- Milvus 文档

LlamaIndex 默认值：
- collection_name：llamacollection
- schema：['doc_id', 'embedding']
- auto_id：True
- {'index_type': 'None', 'metric_type': 'IP',}
- consistency_level：'Strong'
- overwirte：False

In [3]:
# 取消注释以从本地目录读取网页文档。

# Read docs into LlamaIndex
from llama_index.core import SimpleDirectoryReader, StorageContext

# Load HTML files from a local directory
path="RAG/rtdocs_new/"
loader=SimpleDirectoryReader(
    input_dir=path,
    required_exts=[".html"],
    recursive=True  # 递归搜索子目录
)
lli_docs=loader.load_data()

num_documents=len(lli_docs)
print(f"loaded {num_documents} documents")

loaded 22 documents


In [4]:
# Inspect the first document
import pprint

# HTML 文档未被 SimpleDirectoryReader 解析:
    # SimpleDirectoryReader 默认不会解析 HTML 标签
print(f"length doc: {len(lli_docs[0].text)}")
pprint.pprint(lli_docs[0].text[:100])

length doc: 663373
('<!DOCTYPE html><html lang="en"><head><meta charSet="utf-8"/><meta '
 'http-equiv="x-ua-compatible" conte')


In [6]:
from llama_index.core import Settings

from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.milvus import MilvusVectorStore
import time,pprint

# Define the embedding model
Settings.embed_model=HuggingFaceEmbedding(
    model_name="BAAI/bge-large-en-v1.5"
)

# 显示 LlamaIndex 所暴露的内容
print("Embedding model:")
print(Settings.embed_model)
print("Model name:", Settings.embed_model.model_name)

sample_embedding = Settings.embed_model.get_text_embedding("dimension test")
EMBEDDING_DIM = len(sample_embedding)
print(f"EMBEDDING_DIM: {EMBEDDING_DIM}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Embedding model:
model_name='BAAI/bge-large-en-v1.5' embed_batch_size=10 callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x0000023C9E080FB0> num_workers=None embeddings_cache=None rate_limiter=None max_length=512 normalize=True query_instruction=None text_instruction=None cache_folder=None show_progress_bar=False
Model name: BAAI/bge-large-en-v1.5
EMBEDDING_DIM: 1024


In [8]:
from llama_index.core import VectorStoreIndex, StorageContext

# 从文档和嵌入向量创建一个 Milvus 集合
vectorstore=MilvusVectorStore(
    uri="milvus_llamaindex.db",
    dim=EMBEDDING_DIM,
    drop_old=True,
    index_params={
        "metric_type": "COSINE",
        "index_type":"AUTOINDEX",
        "params": {},
    }
)

# 存储上下文
storage_context=StorageContext.from_defaults(
    vector_store=vectorstore,
)

print(f"Start chunking, embedding, inserting...")
start_time = time.time()
llamaindex=VectorStoreIndex.from_documents(
  # Too slow! Just one document
    lli_docs[:1],
    storage_context=storage_context,
)
end_time=time.time()
print(f"Created LlamaIndex collection from {len(lli_docs[:1])} docs in {end_time - start_time:.2f} seconds")


Start chunking, embedding, inserting...
Created LlamaIndex collection from 1 docs in 763.73 seconds
